In [347]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import classification_report,accuracy_score,confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from collections import Counter
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna
import tqdm as notebook_tqdm
import warnings
warnings.filterwarnings('ignore')


In [348]:
model_dt=DecisionTreeClassifier()
model_dt2=DecisionTreeClassifier()
mms=MinMaxScaler()
sc=StandardScaler()
sm=SMOTEENN()
smote=SMOTE(random_state=42)
model_dt_smotenn=DecisionTreeClassifier()
model_rf_smoteenn=RandomForestClassifier(n_estimators=500)
model_xgb=XGBClassifier(random_state=42)
model_xgb_smoteenn=XGBClassifier(random_state=42)
model_xgb_smote=XGBClassifier(random_state=42)
model_rf=RandomForestClassifier(class_weight='balanced',random_state=42,n_estimators=300,max_depth=6)
model_lgb=LGBMClassifier(random_state=42)
model_cat=CatBoostClassifier(auto_class_weights='Balanced',verbose=0,random_state=42)

In [349]:
df=pd.read_csv('../data/Customer-Churn.csv')

In [350]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [351]:
df.Churn.value_counts()/len(df)*100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## **Churn Rate**: 26.53%

- Which means, 26.53% of the customers churn out of this telecom company

In [352]:
y=df['Churn']

x=df.drop(columns=['customerID','Churn'])

print("Shape of x :",x.shape)
print("Shape of y : ",y.shape)

Shape of x : (7043, 19)
Shape of y :  (7043,)


## Train Test Split

In [353]:
x=pd.get_dummies(x,drop_first=True)
y=df['Churn'].map({'No':0,'Yes':1})

In [354]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1)

## Model Building

In [355]:
model_dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [356]:
y_pred_dt=model_dt.predict(x_test)

In [357]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85       522
           1       0.58      0.50      0.53       183

    accuracy                           0.77       705
   macro avg       0.70      0.68      0.69       705
weighted avg       0.77      0.77      0.77       705



In [358]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Initial Insights

- Base model has a accuracy of 76% which is not reliable because of the imbalanced dataset
- TotalCharges needs to be a float/int type (Data Cleaning)
- Need to perform Feature Scaling

## Data Cleaning

In [359]:
telco_data=df.copy()

In [360]:
telco_data.TotalCharges=pd.to_numeric(telco_data.TotalCharges,errors='coerce')

In [361]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [362]:
telco_data.loc[telco_data['TotalCharges'].isnull()==True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [363]:
telco_data.dropna(how='any',inplace=True)

In [364]:
telco_data.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [365]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [366]:
telco_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [367]:
telco_data.tenure.max()

72

In [368]:
bins=[0,12,24,36,48,60,72]

labels=['1-12','13-24','25-36','37-48','40-60','61-72']

telco_data['tenure_bin']=pd.cut(telco_data['tenure'],bins=bins,labels=labels,include_lowest=True)


print(telco_data[['tenure','tenure_bin']].head(10))
print(telco_data['tenure_bin'].value_counts().sort_index())

   tenure tenure_bin
0       1       1-12
1      34      25-36
2       2       1-12
3      45      37-48
4       2       1-12
5       8       1-12
6      22      13-24
7      10       1-12
8      28      25-36
9      62      61-72
tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
40-60     832
61-72    1407
Name: count, dtype: int64


In [369]:
telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [370]:
y=telco_data['Churn']
x=telco_data.drop(columns=['customerID','Churn','tenure'])

print("Shape of X :",x.shape)
print("Shape of y :",y.shape)

Shape of X : (7032, 19)
Shape of y : (7032,)


In [371]:
x

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,tenure_bin
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,1-12
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,25-36
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1-12
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,37-48
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,13-24
7039,Female,0,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,61-72
7040,Female,0,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,1-12
7041,Male,1,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1-12


In [372]:
x=pd.get_dummies(x,drop_first=True)
y=telco_data['Churn'].map({'No':0,'Yes':1})

In [373]:
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_40-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,False,False,True,False,False,False,True,False,True,False,True,False,True,True,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,False,False,False,False,True,False,True,False,False,False,True,False,True,True,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False


In [374]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Feature Scaling

In [375]:
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [376]:
model_dt2.fit(x_train,y_train)
y_pred_dt2=model_dt2.predict(x_test)

In [377]:
print(classification_report(y_test,y_pred_dt2))

              precision    recall  f1-score   support

           0       0.81      0.83      0.82      1054
           1       0.46      0.43      0.45       353

    accuracy                           0.73      1407
   macro avg       0.64      0.63      0.63      1407
weighted avg       0.73      0.73      0.73      1407



## Feature Scaling - MinMaxScaler

In [378]:
x_train_mms=mms.fit_transform(x_train)
x_test_mms=mms.transform(x_test)

In [379]:
model_dt3=DecisionTreeClassifier()
model_dt3.fit(x_train_mms,y_train)

y_pred_dt3=model_dt3.predict(x_test_mms)

In [380]:
print(classification_report(y_test,y_pred_dt3))

              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1054
           1       0.47      0.44      0.46       353

    accuracy                           0.73      1407
   macro avg       0.64      0.64      0.64      1407
weighted avg       0.73      0.73      0.73      1407



## SMOTEENN () [UpSampling + ENN)

In [381]:
x_train_resampled,y_train_resampled=sm.fit_resample(x_train,y_train)

In [382]:
model_dt_smotenn.fit(x_train_resampled,y_train_resampled)
y_pred_dt_smoteenn=model_dt_smotenn.predict(x_test)

In [383]:
print(classification_report(y_test,y_pred_dt_smoteenn))

              precision    recall  f1-score   support

           0       0.88      0.68      0.77      1054
           1       0.43      0.71      0.53       353

    accuracy                           0.69      1407
   macro avg       0.65      0.70      0.65      1407
weighted avg       0.76      0.69      0.71      1407



In [384]:
model_rf_smoteenn.fit(x_train_resampled,y_train_resampled)
y_pred_rf_smoteenn=model_rf_smoteenn.predict(x_test)

In [385]:
print(classification_report(y_test,y_pred_rf_smoteenn))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1054
           1       0.47      0.75      0.58       353

    accuracy                           0.72      1407
   macro avg       0.68      0.73      0.69      1407
weighted avg       0.79      0.72      0.74      1407



## XGBoost without SMOTEENN

In [386]:
model_xgb.fit(x_train,y_train)
y_pred_xgb=model_xgb.predict(x_test)

In [387]:
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1054
           1       0.58      0.51      0.54       353

    accuracy                           0.78      1407
   macro avg       0.71      0.69      0.70      1407
weighted avg       0.78      0.78      0.78      1407



## XGBoost with SMOTEENN

In [388]:
model_xgb_smoteenn.fit(x_train_resampled,y_train_resampled)
y_pred_xgb_smoteenn=model_xgb_smoteenn.predict(x_test)

In [389]:
print(classification_report(y_test,y_pred_xgb_smoteenn))

              precision    recall  f1-score   support

           0       0.89      0.73      0.80      1054
           1       0.48      0.73      0.58       353

    accuracy                           0.73      1407
   macro avg       0.69      0.73      0.69      1407
weighted avg       0.79      0.73      0.75      1407



## XGBoost with SMOTE

In [390]:
print("Before SMOTE: ",Counter(y_train))

x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

print("After SMOTE : ",Counter(y_train_smote))

model_xgb_smote.fit(x_train_smote,y_train_smote)

y_pred_xgb_smote=model_xgb_smote.predict(x_test)

Before SMOTE:  Counter({0: 4109, 1: 1516})
After SMOTE :  Counter({0: 4109, 1: 4109})


In [391]:
print(classification_report(y_test,y_pred_xgb_smote))

              precision    recall  f1-score   support

           0       0.85      0.86      0.86      1054
           1       0.57      0.54      0.56       353

    accuracy                           0.78      1407
   macro avg       0.71      0.70      0.71      1407
weighted avg       0.78      0.78      0.78      1407



## Trying out few more techniques

In [392]:
model_rf.fit(x_train,y_train)
y_pred_rf=model_rf.predict(x_test)
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.73      0.80      1054
           1       0.48      0.76      0.59       353

    accuracy                           0.73      1407
   macro avg       0.69      0.74      0.70      1407
weighted avg       0.79      0.73      0.75      1407



In [393]:
model_cat.fit(x_train,y_train)
y_pred_cat=model_cat.predict(x_test)
print(classification_report(y_test,y_pred_cat))

              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1054
           1       0.51      0.72      0.60       353

    accuracy                           0.76      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.76      0.77      1407



In [394]:
model_lgb.fit(x_train,y_train)
y_pred_lgb=model_lgb.predict(x_test)
print(classification_report(y_test,y_pred_lgb))

[LightGBM] [Info] Number of positive: 1516, number of negative: 4109
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000411 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.269511 -> initscore=-0.997104
[LightGBM] [Info] Start training from score -0.997104
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1054
           1       0.63      0.53      0.58       353

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.80      0.80      0.80      1407



## **Optuna fine tuning**